# Solution Template

Use this notebook as a guide to implement your solution. Keep in mind that some cells should remain as they are so that you code works properly, for instance, the following cell in which the required libraries are imported.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Localization

Hidden Markov Models (HMM) are used in a wide variety of applications such as speech recognition, part-of-speech tagging, machine traslation, handwriting recognition and many more. In this section we will see how HMM can be employed to infer the position of a robot in a static enviroment.

Let us represent the enviroment in which the robot roams around by a grid. Each square of the grid is either free or occupied, in which case we are talking about a position that the robot can not visit. Let $X_k$ be the position of the robot at time $t_k$. Given the latter, all the possible states of $X_k$ are the free spots of the grid: $S\{s_1,s_2,\dots,s_n\}$, where $n$ is the number of free squares. Speaking of "free squares," suppose we do not know where the robot is, so $P(X_0)=1/n$, $\forall s_i\in S$. Also, let $\text{neighnors}(s)$ be the set of empty squares adjacent to $s$ and let $N(s)$ be the size of this set. Then, the transition model of a robot that moves to any adjacent and empty square with equal probability is given by

$$
P(X_{k+1}=s_j|X_k=s_i)=T_{ij}=
\begin{cases}
\frac{1}{N(s_i)}~, & s_j\in \text{neighbors}(s_i)\\
0, & s_j\notin \text{neighbors}(s_i)
\end{cases}
$$

As it was mentioned, the robot will try to know where it is located given a set of observations. In this case, the robot is equipped with four sensors that indicate if there is an obstacle or not in a particular direction. We will assume all the possible directions are North (N), East (E), West (W) and South (S). Further, there is a sensor error rate $\epsilon$ and errors occur independently in the four sensor directions. For instance, the probability of getting all four measurements right is $(1-\epsilon)^4$, the probability of getting them all wrong is $\epsilon^4$. Given this, we have that there are 16 possible measurements for $E_k$ at a given time $t_k$. Also, let $d_{ik}$ be the discrepancy: the number of bits that are different from the true bits for the square $s_i$. Then, the probability that a robot in square $s_i$ would get a reading $e_k$ is equal to

$$
P(E_k=e_k|X_k=s_i)=O_{e_k,ii}=(1-\epsilon)^{4-d_{~ik}}~\epsilon^{~d_{~ik}}.
$$

For instance, the probability that a square with obstacles to the North and South would produce a reading (N,S,E) is equal to $(1-\epsilon)^3~\epsilon$.

Now that we know how to define the transition and sensor models we can carry out different types of inference: we can estimate a location given a set of measurements doing filtering; we can use smoothing to infer a past location given some observations; or we can use the Viterbi algorithm and obtain the most likely path that the robot took and that produced a given set of sensor values.

<img src="https://raw.githubusercontent.com/danotero/MA2014/main/M%C3%B3dulo%20III/robot.png" alt="Original robot map and example posterior probabilities" style="width: 700px;"/>

Your mission is to carry out these three types of inference. In particular, for both filtering and smoothing, instead of using something like the figures shown above, use heatmaps in which the color changes depending on the probability of finding the robot at a certain square given a set of measurements. For the most likely path highlight the squares associated to said path. Keep in mind that you will have to simulate the random behavior of the sensors. 

## Transition Model

Let us define some matrices. Say we have $k$ possible states: $S=\{s_1,s_2\dots,s_k\}$. Then, the **transition matrix** is a $k\times k$ matrix in which $T_{ij} = P(X_{k+1}=s_j|X_{k}=s_i)$. In this case, each state is a position that can be occupied by the robot, or in other words, a free cell. Notice that in the map there are a total of 42 free cells, which means that the system has 42 possible states. The latter implies that the transitions matrix is a $42\times42$ matrix.

Use the following cell to define a dictionary that will contain all the possible states of the system. Remember that each state is a location that can be stored as a tuple. Pick the origin as the upper left corner of the map.

In [ ]:
# Reconstruct the 4 x 16 map shown in the original professor's robot.png.
# '.' is a traversable location; '#' is a wall or unavailable position.
# The state indices follow row-major order, from (0, 0) to (3, 15).
grid = [
    '....#.....#...#.',
    '##..#.##.#.#.###',
    '#...#.##.....##.',
    '..#...#....#....'
]

S = {
    index: (row, column)
    for index, (row, column) in enumerate(
        (row, column)
        for row, cells in enumerate(grid)
        for column, symbol in enumerate(cells)
        if symbol == '.'
    )
}

assert len(S) == 42
assert S[0] == (0, 0) and S[41] == (3, 15)
S

Now that you have the states, write a function that receives as input the dictionary that you just defined, and returns a dictionary in which each element is a state with a list of its corresponding neighbors. This funtion will be useful for implementing both the transition and sensor models.

In [ ]:
def neighbors_states(states, location=True):
    """
    Return the four-directional, obstacle-free neighbors of each state.

    If location=True:  state index -> [(row, column), ...]
    If location=False: state index -> [neighbor state index, ...]
    """
    states_neighbors = {}
    location_to_state = {position: index for index, position in states.items()}
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # E, S, W, N

    for index, (row, column) in states.items():
        positions = [
            (row + dr, column + dc)
            for dr, dc in directions
            if (row + dr, column + dc) in location_to_state
        ]
        states_neighbors[index] = (
            positions if location else [location_to_state[p] for p in positions]
        )

    return states_neighbors

Write a function that computes the transition matrix. The function receives as input the dictionary that contains the states of the system and returns the transition matrix as a `Numpy` array.

In [ ]:
def transition_matrix(states):
    """
    Return the row-stochastic transition matrix T.

    T[i,j] = P(X_(t+1)=j | X_t=i). Each accessible neighbor is equally
    likely. If a free cell has no accessible neighbors, the robot stays there.
    """
    n = len(states)
    t_matrix = np.zeros((n, n), dtype=float)
    neighbors = neighbors_states(states, location=False)

    for origin, adjacent_states in neighbors.items():
        if adjacent_states:
            t_matrix[origin, adjacent_states] = 1.0 / len(adjacent_states)
        else:
            # (0, 15) is isolated on the supplied map.
            t_matrix[origin, origin] = 1.0

    return t_matrix

## Sensor Model

At this point you know that there are 16 possible measurements for $E_k$ at a given time $t_k$. For each measurement you will need to define a matrix that stores the probabilities of getting a reading $e_k$ given that the robot is at the location $s_i$:

$$
P(E_k=e_k|X_k=s_i)=O_{e_k,ii}=(1-\epsilon)^{4-d_{~ik}}~\epsilon^{~d_{~ik}}.
$$

In the following cell, write code for computing the matrices of the sensor model. Notice that you will have to define an error rate $\epsilon$.

In [ ]:
def expected_measurements(states):
    """
    Return each state's noise-free obstacle reading in ESWN order.

    Each bit equals 1 if the corresponding direction is blocked or lies
    outside the 4 x 16 grid; otherwise it equals 0.
    """
    measurements = {}
    traversable = set(states.values())
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # E, S, W, N

    for state, (row, column) in states.items():
        measurements[state] = ''.join(
            '0' if (row + dr, column + dc) in traversable else '1'
            for dr, dc in directions
        )

    return measurements

In [ ]:
def compute_discrepancy(measure_1, measure_2):
    """Return the Hamming distance between two four-bit ESWN readings."""
    if (len(measure_1) != 4 or len(measure_2) != 4
            or not set(measure_1).issubset({'0', '1'})
            or not set(measure_2).issubset({'0', '1'})):
        raise ValueError('Both measurements must be four-character binary strings.')

    discrepancy = sum(a != b for a, b in zip(measure_1, measure_2))
    return discrepancy

In [ ]:
def observation_matrix(epsilon, measure, measurements):
    """
    Return the diagonal matrix of P(sensor reading | robot location).

    Independent sensors each flip their expected bit with probability
    epsilon, so a reading with d wrong bits has likelihood
    (1-epsilon)^(4-d) * epsilon^d.
    """
    if not (0 <= epsilon <= 1):
        raise ValueError('epsilon must be between 0 and 1.')

    n = len(measurements)
    o_matrix = np.zeros((n, n), dtype=float)

    for state, expected in measurements.items():
        d = compute_discrepancy(measure, expected)
        o_matrix[state, state] = (1 - epsilon) ** (4 - d) * epsilon ** d

    return o_matrix

In [ ]:
def observation_matrices(epsilon, states):
    """
    Return all 16 possible sensor matrices, indexed by ESWN bit strings.
    """
    measurements = expected_measurements(states)
    o_matrices = {
        f'{measurement:04b}': observation_matrix(
            epsilon, f'{measurement:04b}', measurements
        )
        for measurement in range(16)
    }

    return o_matrices

Keep in mind that the readings can be understood as binary numbers of four digits: `0000`, `0001`,..., `1111`. Regarding the order of the binary measurements, use this convention: `ESWN`; that is, if the robot has obstacles in both the North and East directions, then an error-free measurement should be the binary number `1001`. For convenience, store the "observation matrices" in a dictionary. 

In [ ]:
"""
Call the function observation_matrices ans store the dictionary it returns in the variable "O". The function
should follow this type of format:

O = {0000: Matrix of measurement 0000, ..., 1111: Matrix of measurement 1111}
"""

epsilon = 0.1

O = observation_matrices(epsilon, S)

In [ ]:
def robot_measurements(state, states):
    """
    Simulate one four-bit ESWN sensor reading at a state.

    Each bit is independently flipped with probability epsilon.
    The 'state' argument can be a numerical state index or a position tuple.
    """
    if isinstance(state, tuple):
        location_to_state = {position: index for index, position in states.items()}
        state = location_to_state[state]

    expected = expected_measurements(states)[state]
    measured = ''.join(
        bit if np.random.random() >= epsilon else ('1' if bit == '0' else '0')
        for bit in expected
    )

    return measured

In [ ]:
def random_robot_walk(states, steps):
    """
    Simulate a random path and noisy sensor evidence.

    path includes the initial location at time zero (length steps+1).
    evidence contains one measurement AFTER each transition (length steps).
    """
    if steps < 1:
        raise ValueError('steps must be at least 1.')

    adjacent = neighbors_states(states, location=False)
    current = int(np.random.choice(list(states.keys())))
    path = [states[current]]
    evidence = []

    for _ in range(steps):
        next_states = adjacent[current]
        if next_states:
            current = int(np.random.choice(next_states))
        # No free neighbor: stay in the current location.
        path.append(states[current])
        evidence.append(robot_measurements(current, states))

    return path, evidence

## Filtering

In this context, **filtering** can be used to get an idea of the robot's location given a set of measurements. As it was mentioned in class, this type of inference is the act of, given some evidence up to a time $t_k$, we want to infer what is the probability distribution of the state $X_k$: $P(X_k|E_{1:k}=e_{1:k})$. Provide the `filtering` function with some evidence, which is going to be a sequence of bynary numbers of four digits, and compute the probability distribution of the state of the system at time $t_k$. Notice that said distribution should be a vector of 42 components. Do not forget to take into account the stochastic nature of the measurement process.

Before going any further, it is important to define first the probability distribution of the initial state $X_0$. Also, run the function `random_robot_walk` to get a a simulation of the random walk of the robot plus a set of measurements. Store this information in tha variables `path` and `E` accordingly. Do that in the next cell.

In [ ]:
# Keep the simulation reproducible so results can be checked.
np.random.seed(2026)

T = transition_matrix(S)
X0 = np.full((len(S), 1), 1.0 / len(S))

# Simulate 20 robot movements and one sensor reading per movement.
path, E = random_robot_walk(S, steps=20)

print('States:', len(S))
print('Transition matrix shape:', T.shape)
print('Uniform starting distribution:', X0.shape)
print('Number of sensor readings:', len(E))
print('Simulated path (including initial location):', path)
print('First five measurements:', E[:5])

Run the next cell for defining the `filtering` function.

In [ ]:
def filtering(E, O, T, X0):
    
    """
    Perform filtering using the Forward algorithm to estimate the probability distribution 
    over states given a sequence of measurements.

    Parameters:
    - E (list): List of binary measurements obtained at each time step.
    - O (dict): Dictionary of observation matrices for different binary measurements.
    - T (numpy.ndarray): Transition matrix representing the probabilities of transitioning between states.
    - X0 (numpy.ndarray): Initial probability distribution over states.

    Returns:
    - forward (numpy.ndarray): Estimated probability distribution over states after incorporating 
                               the sequence of measurements.
    """
    
    forward = X0
    
    for evidence in E:
        forward = O[evidence] @ T.T @ forward
        forward = forward / forward.sum()
        
    return forward

### Visualize

Visualize your results using a heatmap. The `seaborn` library might be very useful for this purpose. In order to do this, write a function that receives the vector $X_k$ and returns a heatmap of a $4\times16$ matrix in which each entry is the probability that the robot is occupying the location associated to said entry.

In [ ]:
def view_heatmap(state, states):
    """
    Display a 4 x 16 heatmap of the 42 robot-state probabilities.

    Blocked cells are gray; only locations present in 'states' are plotted.
    The same helper can display a binary mask of the Viterbi path.
    """
    probabilities = np.asarray(state, dtype=float).reshape(-1)
    if len(probabilities) != len(states):
        raise ValueError('The state vector must have one entry per robot state.')

    map_values = np.full((4, 16), np.nan)
    for index, (row, column) in states.items():
        map_values[row, column] = probabilities[index]

    cmap = plt.cm.viridis.copy()
    cmap.set_bad('lightgray')

    fig, ax = plt.subplots(figsize=(15, 4))
    img = ax.imshow(
        np.ma.masked_invalid(map_values),
        cmap=cmap,
        vmin=0,
        vmax=max(0.01, float(probabilities.max())),
        interpolation='nearest'
    )
    ax.set_xticks(range(16))
    ax.set_yticks(range(4))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_title('Robot localization: probabilities / visited states')
    ax.set_xticks(np.arange(-.5, 16, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which='minor', color='black', linewidth=.7)
    ax.tick_params(which='minor', bottom=False, left=False)
    fig.colorbar(img, ax=ax, label='Probability / path indicator', shrink=.8)
    plt.tight_layout()
    plt.show()

In [ ]:
view_heatmap(filtering(E, O, T, X0), S)

Check the contents of the variable `path`. If there is a reasonable amount of measurements as evidence, the probability of the last element should be the highest. Verify that by running the next cell.

In [ ]:
path

## Smoothing

As for **smoothing**, we can use it to infer a past state given a set of measurements: that is, we wish to obtain $P(X_j|E_{1:k}=e_{1:k})$, $1\leq j<k$. Supply some evidence and run the `smoothing` function. Check that what the functions returns makes sense.

In [ ]:
def smoothing(E, O, T, X0, k):
    
    """
    Perform smoothing using the Forward-Backward algorithm to estimate the probability distribution 
    over states given a sequence of measurements, incorporating both past and future evidence with respect
    to step time k.

    Parameters:
    - E (list): List of binary measurements obtained at each time step.
    - O (dict): Dictionary of observation matrices for different binary measurements.
    - T (numpy.ndarray): Transition matrix representing the probabilities of transitioning between states.
    - X0 (numpy.ndarray): Initial probability distribution over states.
    - k (int): Time step at which to perform smoothing.

    Returns:
    - smooth (numpy.ndarray): Estimated probability distribution over states at time step k, 
                             considering both past and future evidence.
    """
    
    forward = filtering(E[:k], O, T, X0)
    
    backward = np.ones((T.shape[0],1))
    
    for evidence in E[-1:k-1:-1]:
        backward = T @ O[evidence] @ backward
    
    smooth = forward * backward
    smooth = smooth / smooth.sum()
    
    return smooth

In [ ]:
view_heatmap(smoothing(E, O, T, X0, 1), S)

Check the contents of the `path` variable again. If there is a reasonable number of measurements as evidence, the probability of the kth element should be the highest. Check this by executing the next cell.

In [ ]:
path

## Most Likely Sequence of States

Finally, for obtaining the most likely sequence that produced the given evidence, we need to compute

$$
\max_{x_{1:k}}P(x_{1:k}|E_{1:k}=e_{1:k}).
$$

As it was mentioned in class, the most likely sequence is the one that has the highest probability, which is obtained by the **Viterbi algorithm**. 

For this part of the assignment, pick a path in advance and record the set of measurements the robot would take along the road. Do not forget to include the possibility of getting some incorrect measurements in this process.

In [ ]:
def most_likely_sequence(E, O, S, T, X0):
    """
    Run the Viterbi dynamic program and return the most likely locations.

    At every time step, select the most likely predecessor for each
    possible destination, and reconstruct the best path backwards.
    Sensor readings occur after movement, consistent with filtering().
    """
    if not E:
        return []

    n, steps = T.shape[0], len(E)
    prior = np.asarray(X0, dtype=float).reshape(-1)

    scores = np.zeros((n, steps), dtype=float)
    previous = np.zeros((n, steps), dtype=int)

    # First transition from X0, followed by the first observation.
    scores[:, 0] = (T.T @ prior) * np.diag(O[E[0]])

    for time in range(1, steps):
        candidates = scores[:, time - 1, None] * T
        previous[:, time] = np.argmax(candidates, axis=0)
        best_previous = candidates[previous[:, time], np.arange(n)]
        scores[:, time] = best_previous * np.diag(O[E[time]])

        # Rescaling by a positive constant avoids numerical underflow
        # without affecting the optimal path or backpointers.
        scale = scores[:, time].max()
        if scale > 0:
            scores[:, time] /= scale

    current = int(np.argmax(scores[:, -1]))
    indices = [current]
    for time in range(steps - 1, 0, -1):
        current = int(previous[current, time])
        indices.append(current)

    return [S[index] for index in reversed(indices)]

In [ ]:
sequence = most_likely_sequence(E, O, S, T, X0)
sequence

Compare the sequence of states obtained after calling the `most_likely_sequence` function with the actual path followed by the robot. Ignoring the initial position of the robot, i.e. the initial state, all the locations stored in the variable `sequence` should be the same as those stored in the variable `path`.

In [ ]:
path

For visualization purposes, define a vector of 42 components in which the entry associated to a state that belongs to the most likely sequence is set to one, otherwise, set the entry to zero. Call the `view_heatmap` function, which should receive the vector that you just defined as an input, and obtain the corresponding heatmap. The shown path should be the sequence of locations that the robot occupied.

In [ ]:
# Mark each state in the most likely path with 1.
vector = np.zeros((len(S), 1))
location_to_state = {position: index for index, position in S.items()}

for position in sequence:
    vector[location_to_state[position], 0] = 1.0

print('Viterbi locations:', sequence)
print('Simulated locations after movements:', path[1:])
print('Exact position matches:', sum(a == b for a, b in zip(sequence, path[1:])),
      'out of', len(E))

In [ ]:
view_heatmap(vector, S)

## Validation of the HMM

The following small tests check that every state has one outgoing probability
distribution, that all sensor readings form a valid conditional distribution,
and that filtering, smoothing, and Viterbi return correctly shaped results.
Note that noisy sensors may produce a Viterbi path different from the
simulated path; an exact match is not guaranteed.

In [ ]:
# Every row of T must be a probability distribution.
assert T.shape == (42, 42)
assert np.allclose(T.sum(axis=1), 1)
assert np.all(T >= 0)

# The isolated top-right position must keep the robot stationary.
isolated = next(i for i, position in S.items() if position == (0, 15))
assert T[isolated, isolated] == 1.0

# The observation model includes all four-bit patterns.
assert len(O) == 16
assert all(matrix.shape == (42, 42) for matrix in O.values())
assert np.allclose(
    sum(np.diag(matrix) for matrix in O.values()),
    np.ones(42)
)

assert len(path) == len(E) + 1
assert len(sequence) == len(E)
assert all(measure in O for measure in E)
assert all(position in S.values() for position in sequence)

filtered = filtering(E, O, T, X0)
smoothed = smoothing(E, O, T, X0, 1)
assert filtered.shape == (42, 1)
assert smoothed.shape == (42, 1)
assert np.isclose(filtered.sum(), 1)
assert np.isclose(smoothed.sum(), 1)

print('All HMM checks passed.')